Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Evaluate quality

Error rates say whether the agent answered, not whether the answer was good. Amazon Bedrock AgentCore
Evaluations scores traces with evaluators; CloudWatch Omni is where the scores live next to the traces.
You score the step 4 conversation (notebook 05-agent-traces) on demand, then turn on an online evaluation that scores every new
session and publishes each score as a CloudWatch metric.

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 1: The evaluator catalog

In [ ]:
control = boto3.client("bedrock-agentcore-control", region_name=REGION)
evaluations = boto3.client("bedrock-agentcore", region_name=REGION)

catalog = control.list_evaluators()["evaluators"]
builtin = [e for e in catalog if e["evaluatorType"] == "Builtin"]
print(f"{len(builtin)} built-in evaluators (run on service capacity, no model permission needed):")
for e in sorted(builtin, key=lambda e: e["evaluatorId"]):
    print(f"  {e['evaluatorId']:36s} level={e.get('level')}")

## Step 2: Score one turn on demand

`Evaluate` takes the session's span records as input and scores a target trace. The spans come from two
log groups: `aws/spans` (the span tree) and the agent's runtime log group (the prompt and response text the
distro mirrors out as log events). The evaluator needs both.

In [ ]:
def session_spans(session_id, hours_back=3):
    """The session's span records from aws/spans plus the runtime log group (prompt and response text)."""
    runtime_groups = [g["logGroupName"] for g in logs.describe_log_groups(
        logGroupNamePrefix="/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent")["logGroups"]]
    query = (f'fields @message | filter attributes.session.id = "{session_id}"'
             " | filter ispresent(scope.name) and ispresent(traceId) and ispresent(spanId)"
             " | sort @timestamp asc | limit 2000")
    qid = logs.start_query(logGroupNames=["aws/spans"] + runtime_groups,
                           startTime=int(time.time()) - hours_back * 3600,
                           endTime=int(time.time()) + 60, queryString=query)["queryId"]
    while True:
        result = logs.get_query_results(queryId=qid)
        if result["status"] in ("Complete", "Failed", "Cancelled", "Timeout"):
            break
        time.sleep(2)
    return [json.loads(f["value"]) for row in result.get("results", []) for f in row if f["field"] == "@message"]


def invoke_agent_traces(spans):
    return sorted({s["traceId"] for s in spans if str(s.get("name", "")).startswith("invoke_agent")})

In [ ]:
state = load_state()
SESSION_ID = state["OMNI_SESSION_ID"]
spans = session_spans(SESSION_ID)
trace_ids = invoke_agent_traces(spans)
print(f"{len(spans)} spans, {len(trace_ids)} invoke_agent traces")
assert trace_ids, "No invoke_agent trace for this session; run 05-agent-traces again and wait a minute."

target = trace_ids[1] if len(trace_ids) > 1 else trace_ids[0]   # the hotel turn
for evaluator_id in ("Builtin.Helpfulness", "Builtin.ResponseRelevance"):
    response = evaluations.evaluate(evaluatorId=evaluator_id, evaluationInput={"sessionSpans": spans},
                                    evaluationTarget={"traceIds": [target]})
    for r in response["evaluationResults"]:
        if r.get("errorCode"):
            print(f"{evaluator_id}: ERROR {r['errorCode']} {r.get('errorMessage', '')}")
            continue
        print(f"{evaluator_id}: label={r.get('label')} value={r.get('value')}")
        print(f"   {str(r.get('explanation', ''))[:300]}")

Judge-based scores vary a little between runs; read them as a trend, not a unit test. A
`MISSING_OUTPUT_CONTENT` error means the spans carry no response text: the Module 4 span patch was not
applied before deployment.

## Step 3: Turn on continuous scoring

An online evaluation samples live traffic for an agent (selected by service name), runs the evaluators you
choose and writes every score to a results log group and to CloudWatch metrics. `sessionTimeoutMinutes`
decides when a conversation counts as finished: one minute suits a workshop.

In [ ]:
runtime_group = logs.describe_log_groups(
    logGroupNamePrefix="/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent")["logGroups"][0]["logGroupName"]

existing = [c for c in control.list_online_evaluation_configs()["onlineEvaluationConfigs"]
            if c["onlineEvaluationConfigName"] == "workshop_travel_agent"]
if existing:
    EVAL_CONFIG_ID = existing[0]["onlineEvaluationConfigId"]
    print(f"Reusing online evaluation config {EVAL_CONFIG_ID} ({existing[0]['status']}, {existing[0]['executionStatus']})")
else:
    created = control.create_online_evaluation_config(
        onlineEvaluationConfigName="workshop_travel_agent",
        description="Module 5: continuous quality scores for the FAST travel agent",
        rule={"samplingConfig": {"samplingPercentage": 100.0}, "sessionConfig": {"sessionTimeoutMinutes": 1}},
        dataSourceConfig={"cloudWatchLogs": {"logGroupNames": ["aws/spans", runtime_group],
                                             "serviceNames": [state["OMNI_SERVICE_NAME"]]}},
        evaluators=[{"evaluatorId": "Builtin.Helpfulness"}, {"evaluatorId": "Builtin.Refusal"},
                    {"evaluatorId": "Builtin.ToolSelectionAccuracy"}],
        evaluationExecutionRoleArn=state["OMNI_EVAL_ROLE_ARN"],
        outputConfig={"cloudWatchConfig": {"resultDestination": "DEDICATED_LOG_GROUP"}},
        enableOnCreate=True,
    )
    EVAL_CONFIG_ID = created["onlineEvaluationConfigId"]
    print(f"Created online evaluation config {EVAL_CONFIG_ID} ({created['status']}, {created['executionStatus']})")
save_state(OMNI_EVAL_CONFIG_ID=EVAL_CONFIG_ID)

The evaluator list of an enabled configuration is locked so that scores stay comparable over time.

## Step 4: Generate traffic for it to score

The configuration only sees sessions that start after it was enabled. Run one more conversation, then
leave it idle: scoring starts one minute after the last turn and results take roughly ten more minutes.

In [ ]:
import secrets
import urllib.parse

import requests

ssm = boto3.client("ssm", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)


def fast_param(name):
    return ssm.get_parameter(Name=f"/FAST-stack/{name}")["Parameter"]["Value"]


RUNTIME_ARN = fast_param("runtime-arn")
USER_POOL_ID = fast_param("cognito-user-pool-id")
CLIENT_ID = fast_param("cognito-user-pool-client-id")
INVOKE_URL = (f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/"
              f"{urllib.parse.quote(RUNTIME_ARN, safe='')}/invocations?qualifier=DEFAULT")


def access_token_for_throwaway_user():
    """Create a throwaway Cognito user, return (access_token, username).

    The Runtime's JWT authorizer matches the token's client_id claim, so this must
    be the AccessToken, not the IdToken. Delete the user with delete_throwaway_user.
    """
    username = f"omni-{secrets.token_hex(4)}@example.com"
    password = secrets.token_urlsafe(18) + "!Aa1"
    cognito.admin_create_user(
        UserPoolId=USER_POOL_ID, Username=username, MessageAction="SUPPRESS",
        UserAttributes=[{"Name": "email", "Value": username}, {"Name": "email_verified", "Value": "true"}],
    )
    cognito.admin_set_user_password(UserPoolId=USER_POOL_ID, Username=username, Password=password, Permanent=True)
    auth = cognito.initiate_auth(
        AuthFlow="USER_PASSWORD_AUTH", ClientId=CLIENT_ID,
        AuthParameters={"USERNAME": username, "PASSWORD": password},
    )
    return auth["AuthenticationResult"]["AccessToken"], username


def delete_throwaway_user(username):
    cognito.admin_delete_user(UserPoolId=USER_POOL_ID, Username=username)


def ask(token, session_id, prompt):
    """One turn against the runtime. The session ID must be at least 33 characters."""
    response = requests.post(
        INVOKE_URL,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json",
                 "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id},
        json={"prompt": prompt, "runtimeSessionId": session_id},
        timeout=120,
    )
    print(f"HTTP {response.status_code}  <- {prompt[:70]}")
    return response.text

In [ ]:
SCORED_SESSION = f"omni-{secrets.token_hex(16)}-scored"
token, username = access_token_for_throwaway_user()
try:
    ask(token, SCORED_SESSION, "I need a weekend in Tokyo in September for two people, flights from SFO and a hotel")
    ask(token, SCORED_SESSION, "Pick the cheapest flight and a mid-range hotel, and total the trip")
finally:
    delete_throwaway_user(username)
print(f"Scored session: {SCORED_SESSION} (results in ~10 minutes)")

## Step 5: Read the scores as logs and metrics

Results are JSON records in `/aws/bedrock-agentcore/evaluations/results/<config-id>` following the
OpenTelemetry semantic conventions for generative AI evaluation events. Re-run this cell until rows appear.

In [ ]:
results_group = f"/aws/bedrock-agentcore/evaluations/results/{EVAL_CONFIG_ID}"
try:
    events = logs.filter_log_events(logGroupName=results_group, startTime=int(time.time() * 1000) - 3_600_000,
                                    limit=50)["events"]
except logs.exceptions.ResourceNotFoundException:
    events = None
if events is None:
    print("No results group yet: the first scores arrive about ten minutes after a session goes idle.")
elif not events:
    print("Results group exists but is empty so far; try again in a few minutes.")
for e in events or []:
    a = json.loads(e["message"]).get("attributes", {})
    print(f"{str(a.get('gen_ai.evaluation.name', '?')):32s} label={a.get('gen_ai.evaluation.score.label')}"
          f"  value={a.get('gen_ai.evaluation.score.value')}  session=...{str(a.get('session.id', ''))[-8:]}"
          + (f"  error={a['error.type']}" if a.get("error.type") else ""))

cloudwatch = boto3.client("cloudwatch", region_name=REGION)
metrics = cloudwatch.list_metrics(Namespace="Bedrock-AgentCore/Evaluations")["Metrics"]
print(f"\n{len(metrics)} score metrics in Bedrock-AgentCore/Evaluations")
for m in metrics[:12]:
    print(f"  {m['MetricName']:32s} {', '.join(d['Value'] for d in m['Dimensions'])}")

A platform team puts a CloudWatch alarm on a score the same way it alarms on an error rate.

## Step 6: See the scores in Omni

In your space open **Evaluate**, then **Online evaluations**, and select `workshop_travel_agent` for the
trend per evaluator. **Agent traces** now carry the evaluator results on each scored trace.

| Mode | When it runs | Where results go | Use it for |
|------|--------------|------------------|------------|
| On-demand `Evaluate` | When you call it, on spans you supply | The API response | Debugging one run, testing an evaluator's contract |
| Online evaluation | Continuously, on sampled live sessions | Results log group, metrics, Omni trends | Regression detection and alarms |

Continue with `07-regression.ipynb` (optional) or `08-cleanup.ipynb`.